# Build Distance-Based Recovery Access Metrics

This notebook adds distance-based access indicators to the Recovery Access Gap project.

The goal is to move beyond within-municipality service counts and estimate how close each Massachusetts municipality is to source-listed recovery services.

This notebook creates metrics such as:

- nearest source-listed recovery service distance
- services within 5 miles
- services within 10 miles
- service types available within 5 miles
- nearest service distance by service category

These metrics make the dashboard more realistic because recovery access does not stop at municipal boundaries.

In [ ]:
from pathlib import Path

import pandas as pd
import geopandas as gpd
import numpy as np

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
processed_dir = DATA_DIR / "processed"

print("Current working directory:", Path.cwd())
print("Project root:", PROJECT_ROOT)
print("Processed folder:", processed_dir)
print("Processed exists:", processed_dir.exists())

In [ ]:
service_files = {
    "treatment": "samhsa_treatment_facilities_clean.csv",
    "peer_recovery": "peer_recovery_centers_clean.csv",
    "ssp": "syringe_service_programs_clean.csv",
    "harm_reduction": "harm_reduction_programs_deduped_clean.csv",
}

for service_type, filename in service_files.items():
    df = pd.read_csv(processed_dir / filename, dtype={"zip": "string"})
    
    print("\n" + "=" * 80)
    print(service_type.upper())
    print("Rows:", len(df))
    print("Columns:")
    print(df.columns.tolist())
    print("\nPreview:")
    display(df.head(3))

In [ ]:
for service_type, filename in service_files.items():
    df = pd.read_csv(processed_dir / filename, dtype={"zip": "string"})
    
    print("\n" + "=" * 80)
    print(service_type.upper())
    print("Rows:", len(df))
    print("Missing ZIP:", df["zip"].isna().sum())
    print("Unique ZIPs:", df["zip"].nunique())
    print(df[["name", "address", "city", "zip"]].head() if "name" in df.columns else df.head())

In [ ]:
import pgeocode

nomi = pgeocode.Nominatim("us")

all_services = []

for service_type, filename in service_files.items():
    df = pd.read_csv(processed_dir / filename, dtype={"zip": "string"}).copy()
    df["service_type"] = service_type
    df["zip"] = df["zip"].astype("string").str.extract(r"(\d{5})")[0]
    all_services.append(df)

services = pd.concat(all_services, ignore_index=True)

# Remove old coordinate columns if they already exist and are empty/confusing
services = services.drop(
    columns=[col for col in ["latitude", "longitude", "lat", "lon"] if col in services.columns],
    errors="ignore"
)

unique_zips = services["zip"].dropna().unique().tolist()

zip_info = nomi.query_postal_code(unique_zips)

print("pgeocode columns:")
print(zip_info.columns.tolist())

zip_centroids = zip_info[[
    "postal_code",
    "latitude",
    "longitude",
]].copy()

zip_centroids = zip_centroids.rename(columns={"postal_code": "zip"})
zip_centroids["zip"] = zip_centroids["zip"].astype("string").str.zfill(5)

services = services.merge(
    zip_centroids,
    on="zip",
    how="left"
)

print("\nServices columns after merge:")
print(services.columns.tolist())

print("\nMissing latitude:", services["latitude"].isna().sum())
print("Missing longitude:", services["longitude"].isna().sum())

services_geo = gpd.GeoDataFrame(
    services,
    geometry=gpd.points_from_xy(services["longitude"], services["latitude"]),
    crs="EPSG:4326",
)

services_geo = services_geo.to_crs("EPSG:26986")

print("\nTotal service listings:", len(services_geo))
print("Missing ZIP centroid geometry:", services_geo.geometry.isna().sum())

services_geo[[
    "service_type",
    "city",
    "zip",
    "latitude",
    "longitude",
]].head()

In [ ]:
print("Missing ZIP centroid by service type:")
print(
    services.groupby("service_type")[["latitude", "longitude"]]
    .apply(lambda x: x["latitude"].isna().sum())
    .sort_values(ascending=False)
)

print("\nRows with missing latitude/longitude:")
missing_zip_centroids = services[services["latitude"].isna() | services["longitude"].isna()].copy()

print("Missing rows:", len(missing_zip_centroids))
print("Unique missing ZIPs:", missing_zip_centroids["zip"].nunique())

missing_zip_centroids[[
    "service_type",
    "city",
    "state",
    "zip",
]].drop_duplicates().head(50)

In [ ]:
print("ZIP examples by service type:")
for service_type in services["service_type"].unique():
    print("\n", service_type)
    print(services.loc[services["service_type"] == service_type, "zip"].dropna().head(20).tolist())

In [ ]:
for service_type, filename in service_files.items():
    df = pd.read_csv(processed_dir / filename, dtype="string")
    
    print("\n" + "=" * 80)
    print(service_type.upper())
    print("Columns:")
    print(df.columns.tolist())
    
    possible_zip_cols = [col for col in df.columns if "zip" in col.lower() or "postal" in col.lower()]
    print("Possible ZIP columns:", possible_zip_cols)
    
    if possible_zip_cols:
        for col in possible_zip_cols:
            print(f"\n{col} sample:")
            print(df[col].head(10).tolist())
    else:
        print("No ZIP-like columns found.")
    
    display(df.head(3))

In [ ]:
import pgeocode

nomi = pgeocode.Nominatim("us")

all_services = []

for service_type, filename in service_files.items():
    df = pd.read_csv(processed_dir / filename, dtype="string").copy()
    df["service_type"] = service_type
    
    # Extract digits and pad to 5 digits to restore leading zeroes for MA ZIPs
    df["zip"] = (
        df["zip"]
        .astype("string")
        .str.extract(r"(\d+)")[0]
        .str.zfill(5)
    )
    
    all_services.append(df)

services = pd.concat(all_services, ignore_index=True)

# Remove old coordinate columns if they already exist and are empty/confusing
services = services.drop(
    columns=[col for col in ["latitude", "longitude", "lat", "lon"] if col in services.columns],
    errors="ignore"
)

unique_zips = services["zip"].dropna().unique().tolist()

zip_info = nomi.query_postal_code(unique_zips)

zip_centroids = zip_info[[
    "postal_code",
    "latitude",
    "longitude",
]].copy()

zip_centroids = zip_centroids.rename(columns={"postal_code": "zip"})
zip_centroids["zip"] = zip_centroids["zip"].astype("string").str.zfill(5)

services = services.merge(
    zip_centroids,
    on="zip",
    how="left"
)

print("Total service listings:", len(services))
print("Missing latitude:", services["latitude"].isna().sum())
print("Missing longitude:", services["longitude"].isna().sum())

print("\nMissing by service type:")
print(
    services.groupby("service_type")["latitude"]
    .apply(lambda x: x.isna().sum())
    .sort_values(ascending=False)
)

services_geo = gpd.GeoDataFrame(
    services,
    geometry=gpd.points_from_xy(services["longitude"], services["latitude"]),
    crs="EPSG:4326",
)

services_geo = services_geo.to_crs("EPSG:26986")

services_geo[[
    "service_type",
    "city",
    "zip",
    "latitude",
    "longitude",
]].head()

In [ ]:
services_geo_valid = services_geo.dropna(subset=["latitude", "longitude"]).copy()

print("Valid service listings:", len(services_geo_valid))
print("Dropped missing coordinate rows:", len(services_geo) - len(services_geo_valid))

services_geo_valid["service_type"].value_counts()

In [ ]:
towns_geo = gpd.read_file(processed_dir / "ma_municipalities_clean.geojson")

target_crs = "EPSG:26986"

towns_proj = towns_geo.to_crs(target_crs).copy()
towns_proj["town_join"] = towns_proj["TOWN"].astype(str).str.upper().str.strip()

# Use municipality centroid as the representative point for distance calculations
town_centroids = towns_proj[["TOWN", "TOWN_ID", "COUNTY", "town_join", "geometry"]].copy()
town_centroids["geometry"] = town_centroids.geometry.centroid

print("Town centroids:", len(town_centroids))
town_centroids.head()

In [ ]:
nearest_any = gpd.sjoin_nearest(
    town_centroids,
    services_geo_valid[["service_type", "zip", "geometry"]],
    how="left",
    distance_col="nearest_any_service_distance_meters",
)

nearest_any_distance = (
    nearest_any
    .groupby("town_join", as_index=False)["nearest_any_service_distance_meters"]
    .min()
)

nearest_any_distance["nearest_any_service_distance_miles"] = (
    nearest_any_distance["nearest_any_service_distance_meters"] / 1609.344
)

nearest_any_distance.head()

In [ ]:
# Distance thresholds in meters
five_miles_m = 5 * 1609.344
ten_miles_m = 10 * 1609.344

# Cartesian distance join: match each town to services within 10 miles
services_within_10 = gpd.sjoin_nearest(
    town_centroids,
    services_geo_valid[["service_type", "zip", "geometry"]],
    how="left",
    max_distance=ten_miles_m,
    distance_col="distance_to_service_meters",
)

services_within_10["distance_to_service_miles"] = (
    services_within_10["distance_to_service_meters"] / 1609.344
)

print("Town-service matches within 10 miles:", len(services_within_10))

services_within_10[[
    "TOWN",
    "COUNTY",
    "service_type",
    "zip",
    "distance_to_service_miles",
]].head()

In [ ]:
nearby_base = (
    services_within_10
    .dropna(subset=["service_type"])
    .copy()
)

nearby_base["within_5_miles"] = nearby_base["distance_to_service_miles"] <= 5
nearby_base["within_10_miles"] = nearby_base["distance_to_service_miles"] <= 10

# Count total services within 5 and 10 miles
service_counts = (
    nearby_base
    .groupby("town_join", as_index=False)
    .agg(
        services_within_5_miles=("within_5_miles", "sum"),
        services_within_10_miles=("within_10_miles", "sum"),
    )
)

# Count unique service types within 5 miles
types_5 = (
    nearby_base[nearby_base["within_5_miles"]]
    .groupby("town_join")["service_type"]
    .nunique()
    .reset_index(name="service_types_within_5_miles")
)

# Count unique service types within 10 miles
types_10 = (
    nearby_base[nearby_base["within_10_miles"]]
    .groupby("town_join")["service_type"]
    .nunique()
    .reset_index(name="service_types_within_10_miles")
)

nearby_counts = service_counts.merge(types_5, on="town_join", how="left")
nearby_counts = nearby_counts.merge(types_10, on="town_join", how="left")

nearby_counts[
    [
        "services_within_5_miles",
        "services_within_10_miles",
        "service_types_within_5_miles",
        "service_types_within_10_miles",
    ]
] = nearby_counts[
    [
        "services_within_5_miles",
        "services_within_10_miles",
        "service_types_within_5_miles",
        "service_types_within_10_miles",
    ]
].fillna(0).astype(int)

print("Rows with nearby counts:", len(nearby_counts))

nearby_counts.sort_values(
    "services_within_5_miles",
    ascending=False
).head(20)

In [ ]:
distance_access = town_centroids[[
    "TOWN",
    "TOWN_ID",
    "COUNTY",
    "town_join",
]].copy()

distance_access = distance_access.merge(
    nearest_any_distance[[
        "town_join",
        "nearest_any_service_distance_miles",
    ]],
    on="town_join",
    how="left",
)

distance_access = distance_access.merge(
    nearby_counts,
    on="town_join",
    how="left",
)

count_cols = [
    "services_within_5_miles",
    "services_within_10_miles",
    "service_types_within_5_miles",
    "service_types_within_10_miles",
]

distance_access[count_cols] = distance_access[count_cols].fillna(0).astype(int)

print("Distance access rows:", len(distance_access))
print("Municipalities with 0 services within 5 miles:", (distance_access["services_within_5_miles"] == 0).sum())
print("Municipalities with 0 services within 10 miles:", (distance_access["services_within_10_miles"] == 0).sum())

distance_access.sort_values(
    "nearest_any_service_distance_miles",
    ascending=False
).head(20)

In [ ]:
nearest_by_type_frames = []

for service_type in services_geo_valid["service_type"].unique():
    service_subset = services_geo_valid[
        services_geo_valid["service_type"] == service_type
    ][["service_type", "zip", "geometry"]].copy()

    nearest = gpd.sjoin_nearest(
        town_centroids,
        service_subset,
        how="left",
        distance_col="distance_meters",
    )

    nearest_type = (
        nearest
        .groupby("town_join", as_index=False)["distance_meters"]
        .min()
    )

    col_name = f"nearest_{service_type}_distance_miles"
    nearest_type[col_name] = nearest_type["distance_meters"] / 1609.344
    nearest_type = nearest_type[["town_join", col_name]]

    nearest_by_type_frames.append(nearest_type)

nearest_by_type = nearest_by_type_frames[0]

for frame in nearest_by_type_frames[1:]:
    nearest_by_type = nearest_by_type.merge(
        frame,
        on="town_join",
        how="outer",
    )

nearest_by_type.head()

In [ ]:
distance_access = distance_access.merge(
    nearest_by_type,
    on="town_join",
    how="left",
)

distance_access.sort_values(
    "nearest_any_service_distance_miles",
    ascending=False
).head(20)

In [ ]:
distance_access_path = processed_dir / "municipality_distance_access_metrics.csv"

distance_access.to_csv(distance_access_path, index=False)

print("Saved:", distance_access_path)
print("File exists:", distance_access_path.exists())
print("Rows:", len(distance_access))
print("Columns:", len(distance_access.columns))

distance_access.head()

## Join distance access metrics to final priority index

This section adds distance-based service access indicators to the final priority index.

In [ ]:
final_priority = pd.read_csv(processed_dir / "municipality_final_priority_index.csv")

final_priority_with_distance = final_priority.merge(
    distance_access.drop(columns=["TOWN", "TOWN_ID", "COUNTY"], errors="ignore"),
    on="town_join",
    how="left",
)

print("Rows:", len(final_priority_with_distance))
print("Expected municipalities:", len(final_priority))
print("Missing nearest service distance:", final_priority_with_distance["nearest_any_service_distance_miles"].isna().sum())

final_priority_with_distance[[
    "TOWN",
    "COUNTY",
    "recovery_access_gap_score",
    "social_vulnerability_pct",
    "final_priority_score",
    "nearest_any_service_distance_miles",
    "services_within_5_miles",
    "services_within_10_miles",
    "service_types_within_5_miles",
    "service_types_within_10_miles",
]].sort_values("nearest_any_service_distance_miles", ascending=False).head(20)

In [ ]:
# Convert nearby service access into a simple access relief factor
# More nearby service types = more access relief
final_priority_with_distance["nearby_access_coverage_pct"] = (
    final_priority_with_distance["service_types_within_5_miles"] / 4
).clip(0, 1)

# Distance-adjusted priority:
# high original priority remains high, but towns with many nearby service types are slightly reduced
final_priority_with_distance["distance_adjusted_priority_score"] = (
    final_priority_with_distance["final_priority_score"]
    * (1 - 0.35 * final_priority_with_distance["nearby_access_coverage_pct"])
)

final_priority_with_distance[[
    "TOWN",
    "COUNTY",
    "final_priority_score",
    "nearest_any_service_distance_miles",
    "services_within_5_miles",
    "service_types_within_5_miles",
    "nearby_access_coverage_pct",
    "distance_adjusted_priority_score",
]].sort_values(
    "distance_adjusted_priority_score",
    ascending=False
).head(25)

In [ ]:
def assign_distance_access_category(distance):
    if distance <= 2:
        return "Very near"
    elif distance <= 5:
        return "Near"
    elif distance <= 10:
        return "Moderate distance"
    else:
        return "Farther away"


final_priority_with_distance["nearest_service_access_category"] = (
    final_priority_with_distance["nearest_any_service_distance_miles"]
    .apply(assign_distance_access_category)
)

final_priority_with_distance["nearest_service_access_category"].value_counts()

In [ ]:
final_priority_with_distance[[
    "TOWN",
    "COUNTY",
    "nearest_any_service_distance_miles",
    "nearest_service_access_category",
    "services_within_5_miles",
    "services_within_10_miles",
    "service_types_within_5_miles",
    "final_priority_score",
    "distance_adjusted_priority_score",
]].sort_values(
    "nearest_any_service_distance_miles",
    ascending=False
).head(25)

In [ ]:
final_priority_distance_path = processed_dir / "municipality_final_priority_index_with_distance.csv"

final_priority_with_distance.to_csv(final_priority_distance_path, index=False)

print("Saved:", final_priority_distance_path)
print("File exists:", final_priority_distance_path.exists())
print("Rows:", len(final_priority_with_distance))
print("Columns:", len(final_priority_with_distance.columns))

In [ ]:
towns_geo = gpd.read_file(processed_dir / "ma_municipalities_clean.geojson")

towns_geo["town_join"] = towns_geo["TOWN"].astype(str).str.upper().str.strip()

final_priority_distance_geo = towns_geo.merge(
    final_priority_with_distance,
    on="town_join",
    how="left",
    suffixes=("_geo", "")
)

print("Rows:", len(final_priority_distance_geo))
print("Expected municipalities:", len(towns_geo))
print(
    "Missing distance-adjusted score:",
    final_priority_distance_geo["distance_adjusted_priority_score"].isna().sum()
)

final_priority_distance_geo[[
    "TOWN_geo",
    "COUNTY_geo",
    "final_priority_score",
    "distance_adjusted_priority_score",
    "nearest_any_service_distance_miles",
    "services_within_5_miles",
    "service_types_within_5_miles",
]].head()

In [ ]:
final_priority_distance_geo_clean = final_priority_distance_geo.copy()

final_priority_distance_geo_clean["TOWN"] = final_priority_distance_geo_clean["TOWN_geo"]
final_priority_distance_geo_clean["COUNTY"] = final_priority_distance_geo_clean["COUNTY_geo"]

cols_to_drop = [
    "TOWN_geo",
    "COUNTY_geo",
    "TOWN_ID_geo",
]

final_priority_distance_geo_clean = final_priority_distance_geo_clean.drop(
    columns=[col for col in cols_to_drop if col in final_priority_distance_geo_clean.columns]
)

final_priority_distance_geojson_path = (
    processed_dir / "municipality_final_priority_index_with_distance.geojson"
)

final_priority_distance_geo_clean.to_file(
    final_priority_distance_geojson_path,
    driver="GeoJSON"
)

print("Saved:", final_priority_distance_geojson_path)
print("File exists:", final_priority_distance_geojson_path.exists())
print("Rows:", len(final_priority_distance_geo_clean))